# L3 · Recuperación (sub-chunks, embeddings, FAISS, candidatos, pares únicos)

Embeddings **reales** (stage `sandbox`: Ollama `bge-m3` vía `ModelClient.embed`) y **FAISS real** (`IndexFlatIP`); nada simulado.
Entrada: secciones del nivel 1 (`extraer()` + `parsear_bloques()`) para la norma; para los 3 manuales MOCK, **secciones provisionales**
armadas aquí (hojas = encabezados `IV.` y `4.1`) hasta que llegue la cascada de L2. Cuando L2 se integre, el código recibe `list[Seccion]` sin cambios.

Medido con `taskset -c 2,3` y `OMP_NUM_THREADS=2`. Criterio (docs/13): recall@candidatos ≥ 90 % sobre el golden set (arts. 31-48 de la LA/FT frente a los MOCK).

In [ ]:
# --- Arranque por stage (ver 00_modelos.ipynb) ---------------------------------------------------
# Local: el stage sale de config/.env (STAGE=dev MacBook+DMR · STAGE=sandbox Raspberry+Groq+Ollama). Nada que hacer.
# Databricks (STAGE=mvp): descomenta y ejecuta ANTES de importar `backend`.
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()
# import os
# os.environ["STAGE"] = "mvp"
# os.environ["FOUNDRY_AI_TOKEN"] = dbutils.secrets.get(scope="comparador", key="foundry-token")


In [1]:
import os, sys, time, re, csv, json, resource, tempfile, logging
from dataclasses import replace
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

# Ollama compartido en la Pi: con la CPU saturada un texto de ~500 tokens tarda segundos.
# Lotes pequeños y timeout largo SOLO para esta ejecución (variables de entorno, ganan a config).
os.environ.setdefault("EMB_BATCH", "8")
os.environ.setdefault("LLM_TIMEOUT_S", "900")

import numpy as np
from tqdm import tqdm   # barras de progreso
import pymupdf
from backend.config import settings as s
from backend.extraction.docling_extractor import extraer
from backend.extraction.sectioner import parsear_bloques
from backend.models import Seccion
from backend.llm.client import ModelClient
from backend.retrieval.chunker import subchunkear, texto_a_embeber, estimar_tokens
from backend.retrieval.index import construir_indice, embeber, limpiar_cache
from backend.engine.candidatos import candidatos
from backend.engine.pares import construir_pares

from backend.core.progreso import configurar_logs, progreso   # convención: barra de progreso y log sencillo
configurar_logs()
from backend.config import carpeta_normas, carpeta_manuales
NORMAS = carpeta_normas()   # documentos/normas/ (no se versiona); cambia con NORMAS_DIR
if not NORMAS.exists():
    print(f"⚠ Falta la carpeta {NORMAS}: copia ahí los PDF de las normas (ver documentos/LEEME.md)")
LAFIT = NORMAS / "Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf"
GOLDEN = RAIZ / "tests" / "golden_pairs.csv"
MB = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // (1024 if sys.platform.startswith("linux") else 1024 ** 2)
MB_HIJOS = lambda: resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss // (1024 if sys.platform.startswith("linux") else 1024 ** 2)
print("stage:", os.environ.get("STAGE"), "| embeddings:", s.EMB_PROVIDER, s.OLLAMA_EMB_MODEL, "| EMB_BATCH", s.EMB_BATCH)
print(f"SUBCHUNK_TOKENS={s.SUBCHUNK_TOKENS} OVERLAP={s.SUBCHUNK_OVERLAP} K_SUBCHUNKS={s.K_SUBCHUNKS} SIM_THRESHOLD={s.SIM_THRESHOLD} MIN_FLOOR={s.MIN_FLOOR} MAX_CANDIDATES={s.MAX_CANDIDATES}")
print("golden:", GOLDEN.name, "| CPU permitidas:", (sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else f"{os.cpu_count()} (sin afinidad en este SO)"), "| OMP_NUM_THREADS", os.environ.get("OMP_NUM_THREADS"))

stage: sandbox | embeddings: ollama bge-m3 | EMB_BATCH 8
SUBCHUNK_TOKENS=500 OVERLAP=0.15 K_SUBCHUNKS=10 SIM_THRESHOLD=0.5 MIN_FLOOR=3 MAX_CANDIDATES=10
golden: golden_pairs.csv | CPU permitidas: [2, 3] | OMP_NUM_THREADS 2


## 1 · Secciones de la norma (nivel 1, Docling real)
Norma corta `L1-XVI-cap-III.pdf` completa y la LA/FT **solo las páginas 19-24** (arts. 29-54; la RAM compartida no permite pasar las 60 páginas por Docling).
Como las secciones se seleccionan (arts. 31-48), recortar el PDF no cambia los candidatos.

La cascada de L2 (`parsear_bloques`) separa los artículos pegados por Docling.

In [2]:
tmp = tempfile.mkdtemp()
sub = Path(tmp) / "lafit_p19-24.pdf"
orig = pymupdf.open(LAFIT); recorte = pymupdf.open(); recorte.insert_pdf(orig, from_page=18, to_page=23); recorte.save(sub)

BLOQUES = Path(os.environ["BLOQUES_DIR"]) if os.environ.get("BLOQUES_DIR") else None   # opcional: bloques de Docling de una ejecución previa (ahorra ~2 min)
if BLOQUES and (BLOQUES / "cap3.json").exists() and (BLOQUES / "lafit.json").exists():
    b_cap3, b_lafit = (json.load(open(BLOQUES / f"{n}.json")) for n in ("cap3", "lafit")); t_cap3 = t_lafit = 0.0
    print("Bloques reutilizados de", BLOQUES, "(salida de extraer() con Docling real; no se vuelve a extraer)")
else:
    with progreso("Docling · cap-III", "pág") as cb:
        t0 = time.perf_counter(); b_cap3 = extraer(NORMAS / "L1-XVI-cap-III.pdf", cb); t_cap3 = time.perf_counter() - t0
    with progreso("Docling · LA/FT págs 19-24", "pág") as cb:
        t0 = time.perf_counter(); b_lafit = extraer(sub, cb); t_lafit = time.perf_counter() - t0
cap3 = parsear_bloques(b_cap3, "N0", "normativa")
lafit = parsear_bloques(b_lafit, "N1", "normativa")
print(f"Docling: cap-III {len(b_cap3)} bloques (3 págs) · LA/FT p.19-24 {len(b_lafit)} bloques (6 págs) · tiempo de esta ejecución: {t_cap3 + t_lafit:.0f}s")
print("RSS pico Docling (subproceso):", MB_HIJOS(), "MB")
print("cap-III:", [x.identificador for x in cap3])
print("LA/FT:", [x.identificador for x in lafit])

Bloques reutilizados de /tmp/claude-1000/motor-l3 (salida de extraer() con Docling real; no se vuelve a extraer)
Docling: cap-III 43 bloques (3 págs) · LA/FT p.19-24 68 bloques (6 págs) · tiempo de esta ejecución: 0s
RSS pico Docling (subproceso): 52 MB
cap-III: ['ARTÍCULO 1', 'ARTÍCULO 2', 'ARTÍCULO 3', 'ARTÍCULO 4', 'ARTÍCULO 5', 'Artículo 6', 'Artículo 7', 'Artículo 8']
LA/FT: ['Artículo 29', 'Artículo 30', 'Artículo 31', 'Artículo 32', 'Artículo 33', 'Artículo 34', 'Artículo 35', 'Artículo 36', 'Artículo 37', 'Artículo 38', 'Artículo 39', 'Artículo 40', 'Artículo 41', 'Artículo 42', 'Artículo 43', 'Artículo 44', 'Artículo 45', 'Artículo 46', 'Artículo 47', 'Artículo 48', 'Artículo 49', 'Artículo 50', 'Artículo 51', 'Artículo 52', 'Artículo 53', 'Artículo 54']


## 2 · Secciones provisionales de los 3 manuales MOCK
**PROVISIONAL (hasta la cascada de L2):** una `Seccion` por encabezado romano (`V.`) o numeral (`5.3`), con el texto hasta el siguiente encabezado; el índice del PDF se descarta.
`seccionado_incierto=True` para que quede marcado.

In [3]:
ENC = re.compile(r"^(?:(?P<rom>[IVX]+)\.|(?P<num>\d+(?:\.\d+)+))\s+(?P<tit>\S.*)$")

def secciones_provisionales(pdf, doc_id):
    lineas = [(p.number + 1, l.strip()) for p in pymupdf.open(pdf) for l in p.get_text().splitlines() if l.strip()]
    heads = [i for i, (_, l) in enumerate(lineas) if ENC.match(l)]
    primero = lineas[heads[0]][1].split()[0]
    inicio = next(i for i in heads[1:] if lineas[i][1].split()[0] == primero)   # el índice repite los encabezados
    secs, padre, limites = [], None, [i for i in heads if i >= inicio] + [len(lineas)]
    for a, b in zip(limites, limites[1:]):
        m = ENC.match(lineas[a][1]); ident = m["rom"] or m["num"]
        if m["rom"]: padre = f'{ident}. {m["tit"]}'
        secs.append(Seccion(id=f"{doc_id}:{ident}", doc_id=doc_id, tipo_doc="manual_control", nivel="numeral" if m["num"] else "seccion",
                            identificador=ident, titulo=m["tit"], ruta=[padre] if m["num"] else [],
                            texto_literal="\n".join(l for _, l in lineas[a:b]), pagina_inicio=lineas[a][0], pagina_fin=lineas[b - 1][0],
                            seccionado_incierto=True, estrategia="patron", es_hoja=True))
    return secs

NOMBRES = {"N0": "L1-XVI-cap-III", "N1": "Proyecto de Ley LA/FT", "M1": "MOCK-DEMO-01", "M2": "MOCK-DEMO-02", "M3": "MOCK-DEMO-03"}
manual = [x for n in (1, 2, 3) for x in secciones_provisionales(carpeta_manuales() / f"MOCK-DEMO-0{n}.pdf", f"M{n}")]
for n in (1, 2, 3):
    print(f"M{n}:", len([x for x in manual if x.doc_id == f"M{n}"]), "secciones")

M1: 21 secciones
M2: 14 secciones
M3: 24 secciones


## 3 · Sub-chunker (docs/08)

In [4]:
todas = cap3 + lafit + manual
subs = {x.id: subchunkear(x, NOMBRES[x.doc_id]) for x in todas}
n_sub = [len(v) for v in subs.values()]
print(f"{len(todas)} secciones -> {sum(n_sub)} sub-chunks · secciones con más de 1: {sum(n > 1 for n in n_sub)} · máx {max(n_sub)}")
tok = [estimar_tokens(texto_a_embeber(x, c, NOMBRES[x.doc_id])) for x in todas for c in subs[x.id]]
print(f"tokens estimados por texto a embeber: mediana {int(np.median(tok))}, máx {max(tok)} (presupuesto SUBCHUNK_TOKENS={s.SUBCHUNK_TOKENS})")
larga = max(todas, key=lambda x: len(subs[x.id]))
print("Sección más larga:", larga.id, len(larga.texto_literal), "caracteres ->", len(subs[larga.id]), "sub-chunks")
ej = subs[larga.id][1] if len(subs[larga.id]) > 1 else subs[larga.id][0]
print("Texto a embeber (ejemplo):", repr(texto_a_embeber(larga, ej, NOMBRES[larga.doc_id])[:230]))
# 1) sección corta = 1 sub-chunk con el texto literal tal cual; 2) cada sub-chunk es texto de la sección (solo cambian saltos de línea por espacios); 3) el literal no se altera
corta = min(todas, key=lambda x: len(x.texto_literal))
assert len(subs[corta.id]) == 1 and subs[corta.id][0].texto == corta.texto_literal.strip()
norm = lambda t: " ".join(t.split())
assert all(norm(c.texto) in norm(x.texto_literal) for x in todas for c in subs[x.id] if len(subs[x.id]) == 1)
consecutivos = [(norm(a.texto), norm(b.texto)) for x in todas for a, b in zip(subs[x.id], subs[x.id][1:])]
solapes = sum(" ".join(b.split()[:5]) in " ".join(a.split()[-int(s.SUBCHUNK_TOKENS * s.SUBCHUNK_OVERLAP):]) for a, b in consecutivos)
assert consecutivos and solapes == len(consecutivos)   # cada sub-chunk empieza con la cola del anterior
print(f"Chequeos OK: sección corta = 1 sub-chunk literal; {solapes}/{len(consecutivos)} pares consecutivos de sub-chunks con solape")

93 secciones -> 94 sub-chunks · secciones con más de 1: 1 · máx 2
tokens estimados por texto a embeber: mediana 97, máx 466 (presupuesto SUBCHUNK_TOKENS=500)
Sección más larga: N0:LIBRO I/TÍTULO XVI/CAPÍTULO III/SECCIÓN I/ARTICULO-4 4625 caracteres -> 2 sub-chunks
Texto a embeber (ejemplo): 'L1-XVI-cap-III > LIBRO I > TÍTULO XVI > CAPÍTULO III > SECCIÓN I > ARTÍCULO 4: 0.66% | Codificación de las Normas de la Superintendencia de Bancos | 3 | 100.001 | 150.000 | 0.75% | |-----|-----------|-------------|---------| | 2 |'
Chequeos OK: sección corta = 1 sub-chunk literal; 1/1 pares consecutivos de sub-chunks con solape


## 4 · Embeddings reales por lotes + dos índices FAISS
Se indexan la norma (arts. 29-54 de la LA/FT) y los 3 manuales (un solo índice de manual con las secciones de los 3 documentos; el subíndice filtra por selección).

In [5]:
limpiar_cache(); cliente = ModelClient()
t0 = time.perf_counter(); cliente.embed(["calentamiento: carga del modelo en Ollama"]); t_carga = time.perf_counter() - t0
print(f"Primer embedding (carga del modelo): {t_carga:.1f}s")

with progreso("Embeddings · norma LA/FT", "sub-chunk") as cb:
    t0 = time.perf_counter(); i_norma = construir_indice(lafit, cliente, NOMBRES, progreso=cb); t_n = time.perf_counter() - t0
with progreso("Embeddings · manuales MOCK", "sub-chunk") as cb:
    t0 = time.perf_counter(); i_manual = construir_indice(manual, cliente, NOMBRES, progreso=cb); t_m = time.perf_counter() - t0
n_emb = len(i_norma) + len(i_manual)
print(f"norma: {len(i_norma)} sub-chunks en {t_n:.0f}s · manual: {len(i_manual)} sub-chunks en {t_m:.0f}s")
print(f"embeddings/s (textos reales de ~{int(np.mean(tok))} tokens, Ollama compartido): {n_emb / (t_n + t_m):.2f}")
print("dimensión:", i_norma.vectores.shape[1], "· normas L2 ∈", float(np.linalg.norm(i_norma.vectores, axis=1).min()), "-", float(np.linalg.norm(i_manual.vectores, axis=1).max()))
print("FAISS:", type(i_norma.faiss).__name__, i_norma.faiss.ntotal, "|", type(i_manual.faiss).__name__, i_manual.faiss.ntotal, "| mapa subchunk->sección:", len(i_norma.mapa), len(i_manual.mapa))
t0 = time.perf_counter(); construir_indice(manual, cliente, NOMBRES); t_cache = time.perf_counter() - t0
print(f"Reconstruir el índice del manual con la caché por hash: {t_cache * 1000:.0f} ms (0 llamadas al modelo)")
print("RSS pico del notebook:", MB(), "MB")

Primer embedding (carga del modelo): 21.0s


norma: 26 sub-chunks en 113s · manual: 59 sub-chunks en 243s
embeddings/s (textos reales de ~115 tokens, Ollama compartido): 0.24
dimensión: 1024 · normas L2 ∈ 0.9999999403953552 - 1.0000001192092896
FAISS: IndexFlatIP 26 | IndexFlatIP 59 | mapa subchunk->sección: 26 59
Reconstruir el índice del manual con la caché por hash: 2 ms (0 llamadas al modelo)
RSS pico del notebook: 137 MB


## 5 · Candidatos y pares únicos (arts. 31-48 contra los 3 manuales)

In [6]:
num = lambda x: int(m.group()) if x.nivel == "articulo" and (m := re.search(r"\d+", x.identificador)) else 0   # solo artículos
articulos = {x.id for x in lafit if 31 <= num(x) <= 48}
secciones = {x.id for x in manual}
print(len(articulos), "artículos seleccionados ×", len(secciones), "secciones seleccionadas =", len(articulos) * len(secciones), "pares posibles")

# Subíndice filtrado: solo las secciones del MOCK-DEMO-03
sub03 = i_manual.filtrar({x.id for x in manual if x.doc_id == "M3"})
art31 = next(x.id for x in lafit if num(x) == 31)
print("Subíndice filtrado MOCK-DEMO-03:", len(sub03), "sub-chunks · candidatos del Art. 31:", [(a, round(sc, 3)) for a, sc in candidatos(i_norma.vectores_de(art31), sub03)][:5])

t0 = time.perf_counter(); pares, st = construir_pares(i_norma, i_manual, articulos, secciones); t_pares = time.perf_counter() - t0
print(f"Pares con la configuración actual ({t_pares:.2f}s):", st)
print(f"pares ingenuos = {st['n_pares_ingenuo']} · pares únicos = {st['n_pares_unicos']} · ahorro de llamadas al juez: {1 - st['n_pares_unicos'] / st['n_pares_ingenuo']:.0%}")
ej = next(p for p in pares if p.origen == {"v1", "v2"}); print("Par de ejemplo (ambas vías):", ej.model_dump(exclude={"veredicto"}))

18 artículos seleccionados × 59 secciones seleccionadas = 1062 pares posibles
Subíndice filtrado MOCK-DEMO-03: 24 sub-chunks · candidatos del Art. 31: [('M3:I', 0.768), ('M3:II', 0.678), ('M3:V', 0.639), ('M3:5.7', 0.603), ('M3:III', 0.594)]
Pares con la configuración actual (0.01s): {'n_pares_ingenuo': 637, 'n_pares_unicos': 483, 'solo_v1': 20, 'solo_v2': 309, 'ambas': 154}
pares ingenuos = 637 · pares únicos = 483 · ahorro de llamadas al juez: 24%
Par de ejemplo (ambas vías): {'id': 'P1', 'articulo_id': 'N1:CAPITULO III/ARTÍCULO-38', 'seccion_id': 'M3:5.2', 'origen': {'v1', 'v2'}, 'score_v1': 0.7867357730865479, 'rank_v1': 1, 'score_v2': 0.7867357730865479, 'rank_v2': 1, 'cita_norma_verificada': False, 'cita_manual_verificada': False, 'requiere_revision': False}


## 6 · Golden set: recall@candidatos
`recall@candidatos = pares_golden_en_candidatos / total_golden`. Se cuentan las filas del golden con sección esperada (cobertura `cumple`/`parcial`); las filas `omision` no tienen sección y no se pueden recuperar.

In [7]:
ID = {(x.identificador, x.doc_id): x.id for x in lafit + manual}
DOC = {v: k for k, v in NOMBRES.items() if k.startswith("M")}
golden = [r for r in csv.DictReader(open(GOLDEN, encoding="utf-8"))]
con_seccion = [r for r in golden if r["seccion_identificador"]]
print(f"golden: {len(golden)} filas · con sección esperada: {len(con_seccion)} · omisión (sin sección): {len(golden) - len(con_seccion)}")
pares_g = [(ID[(r["articulo_identificador"], "N1")], ID[(r["seccion_identificador"], DOC[r["documento_manual"]])], r) for r in con_seccion]

TODO = replace(s, K_SUBCHUNKS=len(i_manual) + len(i_norma), SIM_THRESHOLD=-1.0, MAX_CANDIDATES=10**6)
def rango(art, sec):  # posición y score del par golden en el ranking completo de cada vía (sin umbral ni tope)
    r1 = [(a, sc) for a, sc in candidatos(i_norma.vectores_de(art), i_manual.filtrar(secciones), TODO)]
    r2 = [(a, sc) for a, sc in candidatos(i_manual.vectores_de(sec), i_norma.filtrar(articulos), TODO)]
    return ([i + 1 for i, (x, _) in enumerate(r1) if x == sec][0], [sc for x, sc in r1 if x == sec][0],
            [i + 1 for i, (x, _) in enumerate(r2) if x == art][0], [sc for x, sc in r2 if x == art][0])

def recall(cfg):
    pr, stt = construir_pares(i_norma, i_manual, articulos, secciones, cfg)
    claves = {(p.articulo_id, p.seccion_id) for p in pr}
    return sum((a, sc) in claves for a, sc, _ in pares_g) / len(pares_g), stt, claves

r, st, claves = recall(s)
print(f"recall@candidatos = {sum((a, sc) in claves for a, sc, _ in pares_g)}/{len(pares_g)} = {r:.1%} (config actual)")
fallos = [(a, sc, row) for a, sc, row in pares_g if (a, sc) not in claves]
for a, sc, row in fallos:
    r1, s1, r2, s2 = rango(a, sc)
    print(f"FALLO {row['articulo_identificador']} -> {row['documento_manual']} {row['seccion_identificador']} ({row['cobertura_esperada']}): "
          f"vía 1 rank {r1} score {s1:.3f} · vía 2 rank {r2} score {s2:.3f}")

golden: 51 filas · con sección esperada: 30 · omisión (sin sección): 21
recall@candidatos = 28/30 = 93.3% (config actual)
FALLO Artículo 31 -> MOCK-DEMO-01 4.1 (parcial): vía 1 rank 36 score 0.493 · vía 2 rank 6 score 0.493
FALLO Artículo 37 -> MOCK-DEMO-01 V (parcial): vía 1 rank 12 score 0.526 · vía 2 rank 11 score 0.526


## 7 · Calibración de `SIM_THRESHOLD` y `K_SUBCHUNKS` (bge-m3: un par sin relación ya da ~0,50)

In [8]:
todos = [candidatos(i_norma.vectores_de(a), i_manual.filtrar(secciones), TODO)[:1][0][1] for a in articulos]
print("Con bge-m3 el mejor score de cada artículo contra el manual va de", round(min(todos), 3), "a", round(max(todos), 3))
sc_golden = [max(rango(a, sec)[1], rango(a, sec)[3]) for a, sec, _ in pares_g]
print("Score (mejor de las dos vías) de los pares golden: min %.3f · mediana %.3f · máx %.3f" % (min(sc_golden), np.median(sc_golden), max(sc_golden)))
filas = []
for k in (5, 10, 20):
    for th in (0.30, 0.50, 0.55, 0.60, 0.65, 0.70):
        rr, stt, _ = recall(replace(s, K_SUBCHUNKS=k, SIM_THRESHOLD=th))
        filas.append((k, th, rr, stt["n_pares_ingenuo"], stt["n_pares_unicos"]))
print(f"{'K':>3} {'umbral':>6} {'recall':>7} {'ingenuo':>8} {'únicos':>7}")
for f in filas: print(f"{f[0]:>3} {f[1]:>6.2f} {f[2]:>7.1%} {f[3]:>8} {f[4]:>7}")

# Valor efectivo de la configuración (STAGE=sandbox -> config/stages/sandbox.env). Un SIM_THRESHOLD en config/.env lo pisaría: se comprueba.
from dotenv import dotenv_values
STAGE_VAL = float(dotenv_values(RAIZ / "config" / "stages" / f"{os.environ['STAGE']}.env")["SIM_THRESHOLD"])
assert s.SIM_THRESHOLD == STAGE_VAL, f"SIM_THRESHOLD efectivo {s.SIM_THRESHOLD} != {STAGE_VAL} del stage: lo pisa config/.env o el entorno"
CAL = s
r_cal, st_cal, claves_cal = recall(CAL)
print(f"Configuración efectiva del stage (SIM_THRESHOLD={CAL.SIM_THRESHOLD}, K_SUBCHUNKS={CAL.K_SUBCHUNKS}): recall@candidatos = {r_cal:.1%} · pares ingenuos {st_cal['n_pares_ingenuo']} · únicos {st_cal['n_pares_unicos']} · {st_cal}")
for a, sec_, row in pares_g:
    if (a, sec_) not in claves_cal:
        r1, s1, r2, s2 = rango(a, sec_)
        print(f"FALLO {row['articulo_identificador']} -> {row['documento_manual']} {row['seccion_identificador']} ({row['cobertura_esperada']}): vía 1 rank {r1} score {s1:.3f} · vía 2 rank {r2} score {s2:.3f}")

Con bge-m3 el mejor score de cada artículo contra el manual va de 0.676 a 0.837
Score (mejor de las dos vías) de los pares golden: min 0.493 · mediana 0.690 · máx 0.837


  K umbral  recall  ingenuo  únicos
  5   0.30   83.3%      385     317
  5   0.50   83.3%      364     296
  5   0.55   83.3%      324     256
  5   0.60   80.0%      285     220
  5   0.65   80.0%      253     202
  5   0.70   80.0%      235     198
 10   0.30   96.7%      770     616
 10   0.50   93.3%      637     483
 10   0.55   86.7%      490     338
 10   0.60   80.0%      344     235
 10   0.65   80.0%      262     203
 10   0.70   80.0%      236     198
 20   0.30   96.7%      770     616
 20   0.50   93.3%      637     483
 20   0.55   86.7%      490     338
 20   0.60   80.0%      344     235
 20   0.65   80.0%      262     203
 20   0.70   80.0%      236     198
Configuración efectiva del stage (SIM_THRESHOLD=0.5, K_SUBCHUNKS=10): recall@candidatos = 93.3% · pares ingenuos 637 · únicos 483 · {'n_pares_ingenuo': 637, 'n_pares_unicos': 483, 'solo_v1': 20, 'solo_v2': 309, 'ambas': 154}
FALLO Artículo 31 -> MOCK-DEMO-01 4.1 (parcial): vía 1 rank 36 score 0.493 · vía 2 rank 6 s

**Lectura de la calibración.** Con bge-m3 el umbral discrimina poco: casi todo par ya supera 0,50, así que el tope `MAX_CANDIDATES` y el piso mandan. El único fallo con el umbral bajo es por **rank 12 > MAX_CANDIDATES (10)**, no por score.
`SIM_THRESHOLD=0.50` es el valor más alto que mantiene recall ≥ 90 % (93,3 %) y recorta pares; 0,55 ya baja a 86,7 %. `K_SUBCHUNKS` 10 y 20 dan lo mismo (cada sección tiene 1-2 sub-chunks): se deja en 10.

## 8 · Norma corta L1-XVI-cap-III contra los manuales (pares sin relación esperada)

In [9]:
with progreso("Embeddings · cap-III", "sub-chunk") as cb:
    i_cap3 = construir_indice(cap3, cliente, NOMBRES, progreso=cb)
p3, st3 = construir_pares(i_cap3, i_manual, {x.id for x in cap3}, secciones)
print("cap-III vs MOCK:", st3)
print("score v1 de sus mejores candidatos: máx %.3f · mediana %.3f (LA/FT 31-48: ver §7)" % (max(p.score_v1 or 0 for p in p3), np.median([p.score_v1 for p in p3 if p.score_v1])))

cap-III vs MOCK: {'n_pares_ingenuo': 231, 'n_pares_unicos': 188, 'solo_v1': 2, 'solo_v2': 143, 'ambas': 43}
score v1 de sus mejores candidatos: máx 0.604 · mediana 0.522 (LA/FT 31-48: ver §7)


## 9 · Hostil: chunker, caché de embeddings, NaN, dimensión
Los casos de error inyectan respuestas malformadas **solo para probar la validación de nuestra capa** (el cliente de prueba no simula recuperación: los índices y embeddings del resto del notebook son reales).

In [10]:
from backend.retrieval import index as idx
from backend.retrieval.index import IndiceError

# Chunker con numerales/literales seguidos (antes: 100 KB 0,3-0,6 s y 400 KB 4-8 s en la Pi)
def hostil(patron, kb):
    txt = patron * (kb * 1024 // len(patron))
    sec = Seccion(id="h", doc_id="d", tipo_doc="normativa", nivel="articulo", identificador="A", ruta=[], texto_literal=txt, pagina_inicio=1, pagina_fin=1, seccionado_incierto=False, estrategia="patron", es_hoja=True)
    t0 = time.perf_counter(); r = subchunkear(sec); return time.perf_counter() - t0, len(r), txt, sec, r
for patron in ("1. ", "a) ", "1.1.1. "):
    for kb in (100, 400):
        seg, n, *_ = hostil(patron, kb)
        print(f"{patron!r:10} {kb:>3} KB -> {seg:.2f}s, {n} sub-chunks")
        if kb == 400: assert seg < 0.5
_, _, txt, sec, r = hostil("1. ", 400)
assert sec.texto_literal == txt                                    # el literal de la sección no se toca
print("literal intacto; sub-chunk máx:", max(estimar_tokens(c.texto) for c in r), "tokens")

# Caché y validación de embeddings (cliente de prueba solo para inyectar respuestas malformadas)
class Falso:
    def __init__(self, resp, modelo="m1"): self.resp, self.modelo, self.s = resp, modelo, s; self.llamadas = 0
    def emb(self): return None, self.modelo
    def embed(self, textos): self.llamadas += 1; return self.resp(len(textos))
buenos = lambda n, d=4: np.tile(np.eye(d, dtype=np.float32)[0], (n, 1))
def intenta(nombre, f):
    try: f(); print("SIN ERROR  ", nombre)
    except IndiceError as e: print(f"{e.codigo} ", nombre, "->", e)
limpiar_cache()
intenta("NaN", lambda: embeber(["a"], Falso(lambda n: np.full((n, 4), np.nan))))
intenta("inf", lambda: embeber(["b"], Falso(lambda n: np.full((n, 4), np.inf))))
intenta("vector cero", lambda: embeber(["c"], Falso(lambda n: np.zeros((n, 4)))))
intenta("respuesta corta", lambda: embeber(["d", "e", "f"], Falso(lambda n: buenos(n - 1))))
assert not idx._CACHE                                              # nada inválido llegó a la caché
c1 = Falso(buenos); embeber(["x"], c1); embeber(["x"], c1); print("misma clave, mismo modelo: llamadas =", c1.llamadas)
c2 = Falso(buenos, "m2"); embeber(["x"], c2); print("otro modelo, mismo texto: llamadas =", c2.llamadas, "(la caché no se mezcla)")
intenta("dimensión distinta, mismo modelo", lambda: embeber(["y"], Falso(lambda n: buenos(n, 8))))
print("embeber([]) ->", embeber([], c1).shape)
idx.MAX_CACHE = 3; embeber([f"t{i}" for i in range(10)], Falso(buenos, "m3")); print("MAX_CACHE=3, lote de 10: entradas en caché =", len(idx._CACHE)); idx.MAX_CACHE = 20000
intenta("Indice con NaN", lambda: idx.Indice([i_norma.subchunks[0]], np.full((1, 4), np.nan)))
intenta("sección inexistente", lambda: i_norma.vectores_de("no-existe"))
intenta("consulta con otra dimensión", lambda: candidatos(np.ones((1, 7), dtype=np.float32), i_norma))
intenta("subíndice vacío", lambda: i_manual.filtrar({"no-existe"}))
limpiar_cache()

'1. '      100 KB -> 0.07s, 110 sub-chunks


'1. '      400 KB -> 0.27s, 438 sub-chunks
'a) '      100 KB -> 0.04s, 94 sub-chunks


'a) '      400 KB -> 0.17s, 374 sub-chunks
'1.1.1. '  100 KB -> 0.03s, 47 sub-chunks
'1.1.1. '  400 KB -> 0.12s, 188 sub-chunks


literal intacto; sub-chunk máx: 495 tokens
ERR-IDX-001  NaN -> El modelo devolvió vectores con NaN, inf o norma cero
ERR-IDX-001  inf -> El modelo devolvió vectores con NaN, inf o norma cero
ERR-IDX-001  vector cero -> El modelo devolvió vectores con NaN, inf o norma cero
ERR-IDX-001  respuesta corta -> El modelo devolvió (2, 4) vectores para 3 textos
misma clave, mismo modelo: llamadas = 1
otro modelo, mismo texto: llamadas = 1 (la caché no se mezcla)
ERR-IDX-001  dimensión distinta, mismo modelo -> Dimensión 8 distinta de la ya vista para sandbox|ollama|m1 (4)
embeber([]) -> (0, 0)
MAX_CACHE=3, lote de 10: entradas en caché = 3
ERR-IDX-001  Indice con NaN -> El modelo devolvió vectores con NaN, inf o norma cero
ERR-IDX-001  sección inexistente -> La sección 'no-existe' no está en el índice
ERR-IDX-001  consulta con otra dimensión -> Consulta (1, 7) incompatible con índice de dimensión 1024
ERR-IDX-001  subíndice vacío -> Ninguna de las secciones seleccionadas está en el índice


## 9b · Manual local (p. ej. ARLAFT) contra la LA/FT 31-48

Misma recuperación, con un manual que **no** es un MOCK y la cascada de seccionado real (sin secciones provisionales). No hay golden set para este manual: se ven los **pares** y, para cada artículo, sus **3 secciones más parecidas** con la ruta, para validarlas a ojo. Va aparte del índice de los MOCK para no alterar la calibración de arriba.

In [ ]:
import pandas as pd
from IPython.display import display
from backend.config import manuales_locales
from backend.extraction.lector import leer_documento
from backend.retrieval.tabla import titulo_visible

BLOQUES_CACHE = Path(os.environ["BLOQUES_DIR"]) if os.environ.get("BLOQUES_DIR") else None
locales = manuales_locales()
if not locales:
    print("No hay manuales locales en documentos/manuales/ (solo los MOCK). Copia ahí tu manual en PDF para verlo en esta sección.")
por_id = {x.id: x for x in lafit}
SALIDA = RAIZ / "output"; SALIDA.mkdir(exist_ok=True)      # carpeta ignorada por git
for k, pdf in enumerate(locales, 1):
    avisos_m = []
    with progreso(f"Docling · {pdf.name}", "pág") as cb:
        secs_m = leer_documento(pdf, f"L{k}", "manual_control", avisos=avisos_m, progreso=cb, cache=BLOQUES_CACHE)
    hojas_m = {x.id for x in secs_m if x.es_hoja}
    with progreso(f"Embeddings · {pdf.stem}", "sub-chunk") as cb:
        i_loc = construir_indice(secs_m, cliente, {**NOMBRES, f"L{k}": pdf.stem}, progreso=cb)
    pares_m, st_m = construir_pares(i_norma, i_loc, articulos, hojas_m)
    print(f"\n=== {pdf.name}: {len(secs_m)} secciones, {len(hojas_m)} hojas, {len(i_loc)} sub-chunks · {len(articulos)} artículos × {len(hojas_m)} hojas = {len(articulos) * len(hojas_m)} pares posibles")
    print(f"pares ingenuos {st_m['n_pares_ingenuo']} → únicos {st_m['n_pares_unicos']} (solo vía 1: {st_m['solo_v1']}, solo vía 2: {st_m['solo_v2']}, ambas: {st_m['ambas']})")
    por_sec = {x.id: x for x in secs_m}
    filas = []
    for a in sorted(articulos, key=lambda i: int(re.search(r"\d+", por_id[i].identificador).group())):
        for rango, (sid, score) in enumerate(candidatos(i_norma.vectores_de(a), i_loc)[:3], 1):
            sc = por_sec[sid]
            filas.append({"Artículo": por_id[a].identificador, "#": rango, "Sección del manual": sc.identificador, "Título": titulo_visible(sc, 55),
                          "Ruta": " › ".join(sc.ruta), "Págs": sc.pagina_inicio, "Score": round(score, 3)})
    tabla_c = pd.DataFrame(filas)
    tabla_c.to_csv(SALIDA / f"L3_candidatos_{pdf.stem}.csv", index=False, encoding="utf-8-sig")
    print(f"\nTop-3 secciones del manual por artículo (score = similitud coseno; umbral {s.SIM_THRESHOLD}):")
    display(tabla_c)

## 10 · Medición y lista de chequeo del criterio de aceptación

In [11]:
print(f"RAM pico del notebook: {MB()} MB · RAM pico de Docling (subproceso): {MB_HIJOS()} MB")
print(f"Tiempos: Docling {t_cap3 + t_lafit:.0f}s · embeddings {t_n + t_m:.0f}s · pares {t_pares:.2f}s")
chequeo = {
  "sub-chunks ~SUBCHUNK_TOKENS, sección corta = 1": max(tok) <= s.SUBCHUNK_TOKENS * 1.1,
  "prefijo de contexto solo en el texto a embeber (literal intacto)": all(texto_a_embeber(x, subs[x.id][0], NOMBRES[x.doc_id]).startswith(NOMBRES[x.doc_id] + " > ") for x in todas),
  "vectores float32 normalizados L2": bool(np.allclose(np.linalg.norm(i_norma.vectores, axis=1), 1, atol=1e-4)) and i_norma.vectores.dtype == np.float32,
  "dos índices FAISS IndexFlatIP con mapa subchunk->sección": i_norma.faiss.ntotal == len(i_norma.mapa) and i_manual.faiss.ntotal == len(i_manual.mapa),
  "caché por hash: segunda construcción sin llamadas al modelo": t_cache < t_m / 20,
  "pares únicos <= pares ingenuos (registro de ambos)": st_cal["n_pares_unicos"] <= st_cal["n_pares_ingenuo"],
  f"recall@candidatos >= 90 % con la configuración calibrada ({r_cal:.1%})": r_cal >= 0.90,
}
for k, v in chequeo.items(): print("[OK]" if v else "[FALLA]", k)
assert all(chequeo.values())

RAM pico del notebook: 157 MB · RAM pico de Docling (subproceso): 52 MB
Tiempos: Docling 0s · embeddings 356s · pares 0.01s
[OK] sub-chunks ~SUBCHUNK_TOKENS, sección corta = 1
[OK] prefijo de contexto solo en el texto a embeber (literal intacto)
[OK] vectores float32 normalizados L2
[OK] dos índices FAISS IndexFlatIP con mapa subchunk->sección
[OK] caché por hash: segunda construcción sin llamadas al modelo
[OK] pares únicos <= pares ingenuos (registro de ambos)
[OK] recall@candidatos >= 90 % con la configuración calibrada (93.3%)
